# Step 2: Ad Generation via LLM API

**Input:** `ad_generation_input.json` (from Step 1)  
**Output:** `ads_llm.json`

**Model:** OpenAI — `gpt-5.4`  
**Generation:** Two-step — (2a) analysis per topic → (2b) A+B generated together with research context + few-shot example

**Strategy A** (strength-amplifying): celebratory tone, draws from positive reviews  
**Strategy B** (problem-solving): reassurance tone, contrast-framing, draws from negative reviews

API calls: 8 analysis + 8 generation = **16 calls total** (same as single-step)

## 1. Install & Import

In [1]:
# %pip install openai --quiet

import json
import time
from openai import OpenAI

In [2]:
from dotenv import load_dotenv
import os


## 2. Configuration

In [3]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL          = "gpt-5.4"

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"Model: {MODEL}")

Model: gpt-5.4


## 3. Load Input Data

In [4]:
with open('ad_generation_input.json', 'r', encoding='utf-8') as f:
    products = json.load(f)

print(f'Products loaded: {len(products)}')
for p in products:
    print(f"  {p['asin']} | {p['product_title'][:55]}")
    for t, d in p['topics'].items():
        print(f"    [{t}]  pos={d['n_positive_total']}  neg={d['n_negative_total']}  "
              f"pct_pos={d['pct_positive']}%")

Products loaded: 3
  B0000775G0 | Woodstock Chimes Home of The Original Guaranteed Musica
    [Product defects / returns]  pos=36  neg=44  pct_pos=45.0%
    [Cymbals]  pos=15  neg=5  pct_pos=75.0%
    [Small percussion accessories]  pos=86  neg=9  pct_pos=90.5%
  B0187KO8X4 | Alesis Nitro Kit | Electronic Drum Set with 8" Snare, 8
    [Product defects / returns]  pos=11  neg=11  pct_pos=50.0%
    [Electronic drums / practice pad]  pos=163  neg=24  pct_pos=87.2%
  B001O5VZ0E | 2.5” Tingsha Bell Cymbals Set - Om Nama Shivay Embossed
    [Product defects / returns]  pos=14  neg=30  pct_pos=31.8%
    [Small percussion accessories]  pos=27  neg=6  pct_pos=81.8%
    [Singing bowls / meditation]  pos=102  neg=7  pct_pos=93.6%


## 4. Prompt Templates (Two-Step)

### Step 2a — Analysis prompt
For each product × topic: ask GPT to extract (a) the specific positive quality to celebrate, and (b) the specific concern + product feature that resolves it.  
This grounds each ad in the actual review content before writing begins.

### Step 2b — Generation prompt
Feed the analysis insights + research context + one few-shot A/B example pair into a single call that produces **both Strategy A and B together**.  
Generating both at once forces the model to actively contrast them.

Key design decisions:
- **Research context** explains *why* A and B must differ psychologically
- **Few-shot example** shows the tone contrast concretely (customer service topic, different product)
- **Combined A+B output** with explicit differentiability requirement prevents convergence
- **Word target 40–55**, 2 sentences each

In [5]:
import re

RESEARCH_CONTEXT = """RESEARCH CONTEXT
----------------
These ads are part of an academic study comparing two advertising persuasion strategies:
  Strategy A — aspiration-driven: celebrate what customers love about this feature.
               Psychological appeal: desire and enthusiasm ("this product is great").
  Strategy B — trust-driven: resolve a specific buyer concern about this feature.
               Psychological appeal: reassurance and confidence ("your worry is answered").
The two ads must be psychologically distinct — a reader seeing only one ad with no label
must immediately know which strategy it represents."""

FEW_SHOT_EXAMPLE = """EXAMPLE (customer service topic — for tone reference only, do not copy content)
-------------------------------------------------------------------------------
Strategy A:
Packed with care and delivered fast, the Crescent MG38-CF arrives intact and exactly as
described—no missing parts, no surprises. Responsive support means any question gets
resolved quickly, making this the confident, easy choice for a first drum kit purchase.

Strategy B:
Not every first drum kit purchase goes smoothly, but Crescent's team responds fast and
resolves any issue before it interrupts your playing. Every MG38-CF ships fully packed
and ready to go—so you can buy with confidence and get straight to learning.

Why these work:
  A opens with a concrete positive (celebratory, aspirational).
  B opens with "Not every..." — implicit contrast that signals the concern without naming it,
  then immediately positions the product as the answer (reassuring, trust-driven).
-------------------------------------------------------------------------------"""


def build_analysis_prompt(product, topic, data):
    """Step 2a: extract the core insight for A and the core concern+feature for B."""
    pos = '\n'.join(f'- {r[:250]}' for r in data['positive_reviews'][:6])
    neg = '\n'.join(f'- {r[:250]}' for r in data['negative_reviews'][:6])
    return f"""You are preparing to write product advertisements. Analyse the reviews below.

PRODUCT: {product['product_title']}
TARGET FEATURE: {topic}

POSITIVE REVIEWS about {topic}:
{pos}

NEGATIVE REVIEWS about {topic}:
{neg}

YOUR TASK
---------
Answer in exactly this two-line format (no extra text):

POSITIVE INSIGHT: [1-2 sentences — what specific quality do satisfied customers praise about {topic}? Be concrete.]
CONCERN AND FEATURE: [1-2 sentences — what specific worry do unhappy customers have about {topic}? Name the ONE product feature that most directly addresses it.]"""


def parse_analysis(text):
    """Parse analysis output into (insight_a, insight_b)."""
    if not text:
        return None, None
    insight_a, insight_b = None, None
    m = re.search(r'POSITIVE INSIGHT:\s*(.+?)(?=CONCERN AND FEATURE:|$)', text, re.DOTALL | re.IGNORECASE)
    if m:
        insight_a = m.group(1).strip()
    m = re.search(r'CONCERN AND FEATURE:\s*(.+?)$', text, re.DOTALL | re.IGNORECASE)
    if m:
        insight_b = m.group(1).strip()
    return insight_a, insight_b


def build_generation_prompt(product, topic, data, insight_a, insight_b):
    """Step 2b: generate both A and B using insights + research context + few-shot."""
    features = '\n'.join(f'- {f}' for f in product['features'][:5]) if product['features'] else 'N/A'
    return f"""You are a senior advertising copywriter writing for an academic A/B study.

{RESEARCH_CONTEXT}

PRODUCT
-------
Name: {product['product_title']}
Brand: {product['brand']}
Description: {product['description'][:500]}
Key features:
{features}
Target feature: {topic}

ANALYSIS INSIGHTS (ground your writing in these)
-------------------------------------------------
Strategy A should celebrate: {insight_a}
Strategy B should resolve:   {insight_b}

{FEW_SHOT_EXAMPLE}

YOUR TASK
---------
Write BOTH ads (40–55 words, 2 sentences each) for the product above.
If a reader sees only one ad with no label, they must immediately know which strategy it is.

Rules for BOTH ads:
- Brand voice only — no customer attribution ("one customer said", "buyers report", etc.)
- One flowing paragraph — no bullet points, no lists
- No percentages, statistics, or counts
- Do NOT use: "outstanding value", "musical journey", "ideal for anyone", "affordable price",
  "beginners and experienced players alike", "take your sound to the next level", "making things right"

Strategy A rules:
- Tone: celebratory and enthusiastic
- Open with a concrete benefit tied to {topic} — not the product name, not "Discover"
- Use ONLY the positive insight above — no negatives or caveats, even indirectly
- Close with a clear reason this is the right choice for {topic}

Strategy B rules:
- Tone: reassuring and confident
- Open with a CONTRAST FRAME that implicitly signals the concern:
    e.g. "Not every [product type] does X — this one does."
    e.g. "[Specific feature] means [concern resolved], so [benefit]."
- NEVER open with: "Play with confidence", "Feel confident", "Rest assured",
  "When X matters", "Not all guitars", or any generic comfort phrase
- Second sentence names the specific product feature that makes the solution work
- Close with what the buyer gains — forward-looking

OUTPUT FORMAT — use exactly:
STRATEGY A:
[ad text]

STRATEGY B:
[ad text]"""


def parse_ads(text):
    """Parse Strategy A and B from combined generation output."""
    if not text:
        return None, None
    ad_a, ad_b = None, None
    m = re.search(r'STRATEGY A:\s*(.+?)(?=STRATEGY B:|$)', text, re.DOTALL | re.IGNORECASE)
    if m:
        ad_a = m.group(1).strip()
    m = re.search(r'STRATEGY B:\s*(.+?)$', text, re.DOTALL | re.IGNORECASE)
    if m:
        ad_b = m.group(1).strip()
    return ad_a, ad_b


print('Two-step prompt functions defined.')
print('\n--- Analysis prompt preview ---')
p0 = products[0]
t0 = list(p0['topics'].keys())[0]
print(build_analysis_prompt(p0, t0, p0['topics'][t0]))


Two-step prompt functions defined.

--- Analysis prompt preview ---
You are preparing to write product advertisements. Analyse the reviews below.

PRODUCT: Woodstock Chimes Home of The Original Guaranteed Musically Tuned Wind Zenergy Hand Chime for Classrooms Meditation Mindfulness and More, Solo
TARGET FEATURE: Product defects / returns

POSITIVE REVIEWS about Product defects / returns:
- I have been using this product in my classroom for over 2 years. It has been abused, pounded on, and dropped, yet it still works perfectly. Great for meditation and any other use you come up with.
- Metal part was a little tilted to one side. Sounds crisp and good. This is not a toy!
- This had a very pretty chime that went on for a second.
- Completely happy with this!
- Missing the rammer
- Exactly as advertised. It has a wonderful sound that reverberates for a good 30 seconds after striking. The mallet is a cheaper plastic on though.

NEGATIVE REVIEWS about Product defects / returns:
- chime is wa

## 5. Generate All 18 Ads (Two-Step)

For each product × topic:
1. **Step 2a** — call analysis prompt → extract `insight_a` and `insight_b`
2. **Step 2b** — call generation prompt with insights → parse both Strategy A and B

Total: 9 analysis calls + 9 generation calls = 18 calls

In [6]:
def call_openai(prompt, max_tokens=400, max_retries=3):
    """Call OpenAI API with retry logic."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_completion_tokens=max_tokens,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = 2 ** attempt
            print(f'    [Attempt {attempt+1} failed: {e} — retrying in {wait}s]')
            time.sleep(wait)
    return None


ads_output = []
ad_count   = 0

for product in products:
    asin = product['asin']
    print(f"\n{'='*65}")
    print(f"{asin} | {product['product_title'][:55]}")

    product_ads = {
        'asin':          asin,
        'product_title': product['product_title'],
        'brand':         product['brand'],
        'ads':           {}
    }

    for topic, data in product['topics'].items():
        print(f"\n  Topic: {topic}")

        # Step 2a: analysis
        print(f"  [2a] Extracting insights...")
        analysis_raw        = call_openai(build_analysis_prompt(product, topic, data), max_tokens=150)
        insight_a, insight_b = parse_analysis(analysis_raw)
        print(f"    A: {insight_a}")
        print(f"    B: {insight_b}")
        time.sleep(1)

        # Step 2b: generation (both A and B in one call)
        print(f"  [2b] Generating A + B...")
        gen_raw    = call_openai(build_generation_prompt(product, topic, data, insight_a, insight_b), max_tokens=400)
        ad_a, ad_b = parse_ads(gen_raw)
        ad_count  += 2
        print(f"  [A] {ad_a}")
        print(f"  [B] {ad_b}")
        time.sleep(1)

        product_ads['ads'][topic] = {
            'strategy_a':   ad_a,
            'strategy_b':   ad_b,
            'insight_a':    insight_a,
            'insight_b':    insight_b,
            'pct_positive': data['pct_positive'],
            'pct_negative': data['pct_negative'],
        }

    ads_output.append(product_ads)

print(f'\nTotal ads generated: {ad_count}')



B0000775G0 | Woodstock Chimes Home of The Original Guaranteed Musica

  Topic: Product defects / returns
  [2a] Extracting insights...
    A: Satisfied customers say the chime is durable and dependable, continuing to work well even after heavy classroom use, drops, and rough handling. They also note that when received in good condition, it sounds crisp and matches the listing.
    B: Unhappy customers worry about receiving a defective or low-quality unit and then being unable to return it. The key feature that addresses this is the product’s return eligibility / return policy.
  [2b] Generating A + B...
  [A] Built to stay dependable through busy classroom days, repeated use, and the occasional drop, this hand chime keeps delivering the crisp, resonant tone you expect. Woodstock Chimes pairs lasting craftsmanship with sound that matches the listing, making it a satisfying choice to keep and enjoy.
  [B] Not every hand chime arrives exactly as hoped, but this one is backed to keep your

## 6. Save Output

In [7]:
with open('ads_llm.json', 'w', encoding='utf-8') as f:
    json.dump(ads_output, f, indent=2, ensure_ascii=False)

print('Saved to ads_llm.json')

print('\n--- Sanity Check ---')
for p in ads_output:
    print(f"\n{p['asin']} | {p['product_title'][:55]}")
    for topic, ads in p['ads'].items():
        a_ok = '✓' if ads['strategy_a'] else '✗ MISSING'
        b_ok = '✓' if ads['strategy_b'] else '✗ MISSING'
        print(f"  [{topic}]  A:{a_ok}  B:{b_ok}")


Saved to ads_llm.json

--- Sanity Check ---

B0000775G0 | Woodstock Chimes Home of The Original Guaranteed Musica
  [Product defects / returns]  A:✓  B:✓
  [Cymbals]  A:✓  B:✓
  [Small percussion accessories]  A:✓  B:✓

B0187KO8X4 | Alesis Nitro Kit | Electronic Drum Set with 8" Snare, 8
  [Product defects / returns]  A:✓  B:✓
  [Electronic drums / practice pad]  A:✓  B:✓

B001O5VZ0E | 2.5” Tingsha Bell Cymbals Set - Om Nama Shivay Embossed
  [Product defects / returns]  A:✓  B:✓
  [Small percussion accessories]  A:✓  B:✓
  [Singing bowls / meditation]  A:✓  B:✓


## 7. Full Preview

In [8]:
with open('ads_llm.json', 'r', encoding='utf-8') as f:
    ads_output = json.load(f)

for p in ads_output:
    print(f"\n{'='*65}")
    print(f"{p['asin']} | {p['product_title']}")
    for topic, ads in p['ads'].items():
        a_words = len(ads['strategy_a'].split()) if ads['strategy_a'] else 0
        b_words = len(ads['strategy_b'].split()) if ads['strategy_b'] else 0
        print(f"\n  [{topic}]  ({ads['pct_positive']}% pos / {ads['pct_negative']}% neg)")
        print(f"  Insight A: {ads.get('insight_a', 'N/A')}")
        print(f"  Insight B: {ads.get('insight_b', 'N/A')}")
        print(f"  Strategy A ({a_words}w): {ads['strategy_a']}")
        print(f"  Strategy B ({b_words}w): {ads['strategy_b']}")



B0000775G0 | Woodstock Chimes Home of The Original Guaranteed Musically Tuned Wind Zenergy Hand Chime for Classrooms Meditation Mindfulness and More, Solo

  [Product defects / returns]  (45.0% pos / 55.0% neg)
  Insight A: Satisfied customers say the chime is durable and dependable, continuing to work well even after heavy classroom use, drops, and rough handling. They also note that when received in good condition, it sounds crisp and matches the listing.
  Insight B: Unhappy customers worry about receiving a defective or low-quality unit and then being unable to return it. The key feature that addresses this is the product’s return eligibility / return policy.
  Strategy A (45w): Built to stay dependable through busy classroom days, repeated use, and the occasional drop, this hand chime keeps delivering the crisp, resonant tone you expect. Woodstock Chimes pairs lasting craftsmanship with sound that matches the listing, making it a satisfying choice to keep and enjoy.
  Strategy B 